# Verification of `CorotationalFEMForceField`

This notebook verifies SOFA's `CorotationalFEMForceField` with the Method of Manufactured Solutions, presented in [IntroductionMMS.ipynb](IntroductionMMS.ipynb). It presents the governing PDEs and some of the cases used for verification, in `verification/cases/CorotationalFEMForceField/`.

## Governing PDE for Corotational Formulation of Linear Elastic Materials

The corotational formulation extends the linear elastic material of [LinearSmallStrainFEMForceField.ipynb](LinearSmallStrainFEMForceField.ipynb) to large rotations [[1]](#ref1). Throughout, $\mathbf{x}$ denotes the rest coordinates, $\nabla$ the gradient with respect to $\mathbf{x}$, and $\mathbf{F} = \mathbf{I} + \nabla \mathbf{u}$ the deformation gradient.

The linear material measures strain with $\boldsymbol{\varepsilon}(\mathbf{u}) = \tfrac12(\nabla \mathbf{u} + \nabla \mathbf{u}^T)$, which mistakes rotation for strain. A rigid rotation $\mathbf{u} = \mathbf{R}\,\mathbf{x} - \mathbf{x}$, by a constant angle $\theta$ in 2D, gives

$$\boldsymbol{\varepsilon}(\mathbf{u}) = \tfrac12(\mathbf{R} + \mathbf{R}^T) - \mathbf{I} = (\cos\theta - 1)\,\mathbf{I}$$

The body has not deformed, yet the linear material sees a uniform compression of $1 - \cos\theta$: about $\theta^2/2$ for small angles, and $100\%$ at $\theta = 90^\circ$.

### Polar Decomposition

Any deformation gradient with $\det \mathbf{F} > 0$ splits uniquely into a rotation and a stretch:

$$\mathbf{F} = \mathbf{R}\,\mathbf{U}, \qquad \mathbf{R}^T \mathbf{R} = \mathbf{I}, \quad \det \mathbf{R} = 1, \qquad \mathbf{U} = \mathbf{U}^T \text{ positive definite}$$

$\mathbf{U}$ carries the change of shape and $\mathbf{R}$ the orientation. In general $\mathbf{R}(\mathbf{x})$ is a field, but it cannot vary freely: $\mathbf{F} = \mathbf{R}(\mathbf{x})$ with $\mathbf{U} = \mathbf{I}$ is the gradient of a displacement only if $\mathbf{R}$ is constant. A varying rotation therefore always comes with strain. For small $\nabla \mathbf{u}$, $\mathbf{U} - \mathbf{I}$ reduces to $\boldsymbol{\varepsilon}(\mathbf{u})$: the linear material is the small-rotation limit.

### Corotational Law

Once the rotation is known, the strain is pulled back to the unrotated frame, the linear law is applied to it, and the stress is turned back with $\mathbf{R}$:

$$\boldsymbol{\varepsilon}_U = \mathbf{R}^T \mathbf{F} - \mathbf{I} = \mathbf{U} - \mathbf{I}, \qquad \psi_U = \psi(\boldsymbol{\varepsilon}_U), \qquad \mathbf{P} = \frac{\partial \psi_U}{\partial \mathbf{F}} = \mathbf{R}\,\boldsymbol{\sigma}(\boldsymbol{\varepsilon}_U), \qquad \boldsymbol{\sigma}(\boldsymbol{\varepsilon}) = \lambda\,\mathrm{tr}(\boldsymbol{\varepsilon})\,\mathbf{I} + 2\mu\,\boldsymbol{\varepsilon}$$

with the strain energy density $\psi$ and the Lamé parameters $\lambda$ and $\mu$ of the linear material. Strain, energy and stress are those of the linear material, evaluated on the stretch alone. The last equality holds because $\boldsymbol{\sigma}(\boldsymbol{\varepsilon}_U)$ commutes with $\mathbf{U}$, so the terms from differentiating $\mathbf{R}$ cancel. A rigid rotation gives $\boldsymbol{\varepsilon}_U = \mathbf{0}$, and small rotations recover the linear material. The displacement satisfies

$$\mathcal{L}(\mathbf{u}) = -\nabla \cdot \mathbf{P}(\mathbf{u}) = \mathbf{s}$$

with $\mathbf{s}$ a force per unit rest volume. This is the equation solved by the `CorotationalFEMForceField` component in SOFA. It computes one rotation per element, approximating $\mathbf{R}$, and pulls the element's current node positions back by it before computing anything. The strain, stress and energy it computes contain the stretch only, and only the forces are turned back.

### Manufactured Solution

A manufactured solution $\hat{\mathbf{u}}_R$, with $\det \hat{\mathbf{F}} > 0$, determines all the references, in order: its deformation gradient, the rotation and stretch of that gradient, then the pulled-back strain, energy and stress, and the source term:

$$\hat{\mathbf{F}} = \mathbf{I} + \nabla \hat{\mathbf{u}}_R = \hat{\mathbf{R}}\,\hat{\mathbf{U}}, \qquad \hat{\boldsymbol{\varepsilon}}_U = \hat{\mathbf{R}}^T \hat{\mathbf{F}} - \mathbf{I}, \qquad \hat{\psi}_U = \psi(\hat{\boldsymbol{\varepsilon}}_U), \qquad \hat{\mathbf{P}}_R = \hat{\mathbf{R}}\,\boldsymbol{\sigma}(\hat{\boldsymbol{\varepsilon}}_U), \qquad \mathbf{s}_R = -\nabla \cdot \hat{\mathbf{P}}_R$$

The source term is built from $\hat{\mathbf{u}}_R$ through the corotational law itself, so $\hat{\mathbf{u}}_R$ solves $\mathcal{L}(\mathbf{u}) = \mathbf{s}_R$ exactly, with its own values on the boundary. A correct discretisation of the corotational law converges to $\hat{\mathbf{u}}_R$, and one that treats rotation as strain does not.

The cases build $\hat{\mathbf{u}}_R$ from a manufactured solution $\hat{\mathbf{u}}$ of the linear notebook and a generator rotation $\mathbf{R}$: the body is deformed by $\hat{\mathbf{u}}$, then rotated about the origin,

$$\hat{\mathbf{u}}_R = \mathbf{R}\,(\mathbf{x} + \hat{\mathbf{u}}) - \mathbf{x}$$

$\mathbf{R}$ is a parameter of the manufactured solution, like the amplitude of $\hat{\mathbf{u}}$. It is in general not the rotation $\hat{\mathbf{R}}$ of $\hat{\mathbf{F}}$, which the references above use.

#### Constant Rotation

With a constant $\mathbf{R}$, $\hat{\mathbf{F}} = \mathbf{R}\,(\mathbf{I} + \nabla \hat{\mathbf{u}})$. When $\nabla \hat{\mathbf{u}}$ is symmetric, as for the trigonometric function, and $\mathbf{I} + \nabla \hat{\mathbf{u}}$ is positive definite, $\mathbf{I} + \nabla \hat{\mathbf{u}}$ is already the stretch. The polar decomposition then returns the generator, $\hat{\mathbf{R}} = \mathbf{R}$, and the references reduce to

$$\hat{\boldsymbol{\varepsilon}}_U = \boldsymbol{\varepsilon}(\hat{\mathbf{u}}), \qquad \hat{\psi}_U = \psi(\boldsymbol{\varepsilon}(\hat{\mathbf{u}})), \qquad \hat{\mathbf{P}}_R = \mathbf{R}\,\hat{\boldsymbol{\sigma}}, \qquad \mathbf{s}_R = \mathbf{R}\,\mathbf{s}$$

with $\hat{\boldsymbol{\sigma}}$ and $\mathbf{s}$ the stress and source term of $\hat{\mathbf{u}}$ in the linear notebook. The rotated problem is the unrotated linear problem turned by $\mathbf{R}$: pulled back, its strain, energy and stress are those of linear elasticity in the unrotated frame. `LinearSmallStrainFEMForceField` on the rotated problem sees the strain $\mathrm{sym}(\mathbf{R}\,(\mathbf{I} + \nabla \hat{\mathbf{u}})) - \mathbf{I}$, which contains the spurious $(\cos\theta - 1)\,\mathbf{I}$ above, so its errors stop decreasing.

#### Varying Rotation

With a rotation field $\mathbf{R}(\theta(\mathbf{x}))$, the gradient gains a term from the varying angle,

$$\hat{\mathbf{F}} = \mathbf{R}\,(\mathbf{I} + \nabla \hat{\mathbf{u}}) + \frac{\partial \mathbf{R}}{\partial \theta}\,(\mathbf{x} + \hat{\mathbf{u}}) \otimes \nabla \theta$$

This term is the strain that neighbouring points turning by different angles must cause. $\mathbf{R}^T \hat{\mathbf{F}}$ is no longer symmetric, so $\mathbf{R}(\theta)$ is not the rotation of $\hat{\mathbf{F}}$. $\hat{\mathbf{R}}$ is computed from $\hat{\mathbf{F}}$ by polar decomposition, in closed form in 2D:

$$\hat{\mathbf{R}} = \frac{1}{\sqrt{a^2 + b^2}}\begin{pmatrix} a & -b \\ b & a \end{pmatrix}, \qquad a = \hat{F}_{11} + \hat{F}_{22}, \qquad b = \hat{F}_{21} - \hat{F}_{12}$$

and the references follow as they stand. $\mathbf{R}(\theta)$ plays no further role.

In 3D the polar decomposition has no closed form that can be differentiated. $\hat{\mathbf{R}}$ is computed at each point from the singular value decomposition $\hat{\mathbf{F}} = \mathbf{W}\,\boldsymbol{\Sigma}\,\mathbf{V}^T$, as $\hat{\mathbf{R}} = \mathbf{W}\,\mathbf{V}^T$, and $\mathbf{s}_R$ by fourth-order central differences of $\hat{\mathbf{P}}_R$.

For example, with $\hat{\mathbf{u}} = \mathbf{0}$ and $\theta = \alpha x$,

$$\mathbf{R}^T \hat{\mathbf{F}} = \begin{pmatrix} 1 - \alpha y & 0 \\ \alpha x & 1 \end{pmatrix}, \qquad \det \hat{\mathbf{F}} = 1 - \alpha y$$

The strain is of order $\alpha\,|\mathbf{x}|$: it grows with the distance from the origin. At $\alpha = 45^\circ$ per unit length and $\mathbf{x} = (0.7, 0.4)$, the generator turns by $31.5^\circ$, but $\hat{\mathbf{R}}$ by $49.6^\circ$. The rates and the domain must keep $\det \hat{\mathbf{F}} > 0$ everywhere.

### Solving and Measuring

`CorotationalFEMForceField` solves the rotated problem, starting from the rotated rest configuration $\mathbf{R}\,\mathbf{x}$, with the source term, the boundary tractions and the boundary displacements applied at their full values in one load step. Its tangent accounts for the change of each element's rotation, which makes it not symmetric, so the linear solves use `EigenBiCGSTAB`. The errors are measured against the references above.

- **$L^2$ error.** $\|\mathbf{u}_h - \hat{\mathbf{u}}_R\|$ as it is: SOFA stores total positions, so $\mathbf{u}_h$ holds the rotation, as $\hat{\mathbf{u}}_R$ does.
- **$H^1$ error.** The gradients are pulled back with $\hat{\mathbf{R}}$, $\nabla \mathbf{u} \mapsto \hat{\mathbf{R}}^T (\mathbf{I} + \nabla \mathbf{u}) - \mathbf{I}$. This leaves the error unchanged, since $|\hat{\mathbf{R}}^T \mathbf{A}| = |\mathbf{A}|$.
- **Energy.** The energy SOFA reports for $\mathbf{u}_h$ is already pulled back, element by element. It is compared with $\int \hat{\psi}_U$: with a constant rotation, the energy of linear elasticity in the unrotated frame.

## Running the Cases

The cell below loads the suite's runner and prepares the figures. Run it once before any case: every case cell below then runs on its own, in any order.

In [ ]:
# Make the plugin's packages importable from verification/docs
import sys
sys.path.insert(0, "../..")

# Hide the runner's printed report; figures still show
import os
sys.stdout = open(os.devnull, "w")

# Load the suite's runner
from verification.run import run

# For showing figures inline
%matplotlib inline

# For drawing several convergence figures side by side
import matplotlib.pyplot as plt

## Triangle and Quadrilateral Elements on a 2D Beam

The beam starts at the origin, with length $L$ along $x$ and width $W$ along $y$.

### Trigonometric Function

$$\hat{\mathbf{u}}_R = \mathbf{R}\,(\mathbf{x} + \hat{\mathbf{u}}) - \mathbf{x}, \qquad \hat{\mathbf{P}}_R = \mathbf{R}\,\hat{\boldsymbol{\sigma}}, \qquad \mathbf{s}_R = 2k^2(\lambda + 2\mu)\,\mathbf{R}\,\hat{\mathbf{u}}, \qquad \mathbf{R} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$$

$$\hat{\mathbf{u}} = A\begin{pmatrix} \sin(kx)\cos(ky) \\ \cos(kx)\sin(ky) \end{pmatrix}, \qquad \hat{\boldsymbol{\sigma}} = 2Ak\begin{pmatrix} (\lambda + \mu)\cos(kx)\cos(ky) & -\mu\sin(kx)\sin(ky) \\ -\mu\sin(kx)\sin(ky) & (\lambda + \mu)\cos(kx)\cos(ky) \end{pmatrix}$$

with $k = \frac{2\pi}{L}$, $L = W$, and $\theta = 45^\circ$.

- `left` ($x = 0$), `bottom` ($y = 0$): $\mathbf{u} = \hat{\mathbf{u}}_R$
- `right` ($x = L$), `top` ($y = L$): $\mathbf{t} = \hat{\mathbf{P}}_R\,\mathbf{n}$

`LinearSmallStrainFEMForceField` runs first on the unrotated problem with triangles, as the reference for the deformed shapes below, then `CorotationalFEMForceField` on the rotated one.

In [ ]:
# Reference: 2D rectangle, unrotated, trigonometric manufactured solution
linear_tri = run("../cases/LinearSmallStrainFEMForceField/Trigonometric2D_Tri.json")

# Case: 2D rectangle rotated by 45°, trigonometric manufactured solution, triangles (left) and quadrilaterals (right)
tri = run("../cases/CorotationalFEMForceField/Trigonometric2D_Tri_ConstantRotation.json")
quad = run("../cases/CorotationalFEMForceField/Trigonometric2D_Quad_ConstantRotation.json")
figure, (triangles, quadrilaterals) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tri.plot(triangles)
quad.plot(quadrilaterals);

Deformed shapes, the reference on triangles first, then the corotational ones on triangles and quadrilaterals. The corotational shapes are drawn turned by $\mathbf{R}$, as solved, and coloured by the displacement pulled back to the unrotated frame: the colours match the reference.

In [ ]:
# Deformed shape at each level, coarse to fine: reference, then corotational
linear_tri.plot_fields()
tri.plot_fields()
quad.plot_fields()

`LinearSmallStrainFEMForceField` on the same rotated problem, quadrilaterals: its errors stop decreasing.

In [ ]:
# Case: the quadrilateral case above, with LinearSmallStrainFEMForceField
from verification.case import Case
from verification.study.study import ErrorConvergenceStudy

case = Case.load("../cases/CorotationalFEMForceField/Trigonometric2D_Quad_ConstantRotation.json")
case.force_field = {"type": "LinearSmallStrainFEMForceField"}
failure = ErrorConvergenceStudy(case)
failure.run()

### Trigonometric Function with a Varying Rotation

$$\hat{\mathbf{u}}_R = \mathbf{R}(\theta)\,(\mathbf{x} + \hat{\mathbf{u}}) - \mathbf{x}, \qquad \theta = \alpha_x x + \alpha_y y, \qquad \hat{\boldsymbol{\varepsilon}}_U = \hat{\mathbf{R}}^T \hat{\mathbf{F}} - \mathbf{I}, \qquad \hat{\mathbf{P}}_R = \hat{\mathbf{R}}\,\boldsymbol{\sigma}(\hat{\boldsymbol{\varepsilon}}_U), \qquad \mathbf{s}_R = -\nabla \cdot \hat{\mathbf{P}}_R$$

$$\hat{\mathbf{F}} = \mathbf{I} + \nabla \hat{\mathbf{u}}_R, \qquad \hat{\mathbf{R}} = \frac{1}{\sqrt{a^2 + b^2}}\begin{pmatrix} a & -b \\ b & a \end{pmatrix}, \qquad a = \hat{F}_{11} + \hat{F}_{22}, \qquad b = \hat{F}_{21} - \hat{F}_{12}$$

with $\hat{\mathbf{u}}$ and $\mathbf{R}$ as in the trigonometric case above, and $\alpha_x = 7.5^\circ$, $\alpha_y = 5^\circ$ per unit length.

- `left` ($x = 0$), `bottom` ($y = 0$): $\mathbf{u} = \hat{\mathbf{u}}_R$
- `right` ($x = L$), `top` ($y = L$): $\mathbf{t} = \hat{\mathbf{P}}_R\,\mathbf{n}$

`LinearSmallStrainFEMForceField` on the same problem is the control: its errors stop decreasing.

In [ ]:
# Case: 2D rectangle with a rotation field, trigonometric manufactured solution, triangles (left) and quadrilaterals (right)
tri_field = run("../cases/CorotationalFEMForceField/Trigonometric2D_Tri_VaryingRotation.json")
field = run("../cases/CorotationalFEMForceField/Trigonometric2D_Quad_VaryingRotation.json")
figure, (triangles, quadrilaterals) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tri_field.plot(triangles)
field.plot(quadrilaterals);

# Deformed shape at each level, coarse to fine
tri_field.plot_fields()
field.plot_fields()

In [ ]:
# Control: the same case, with LinearSmallStrainFEMForceField
from verification.case import Case
from verification.study.study import ErrorConvergenceStudy

case = Case.load("../cases/CorotationalFEMForceField/Trigonometric2D_Quad_VaryingRotation.json")
case.force_field = {"type": "LinearSmallStrainFEMForceField"}
control = ErrorConvergenceStudy(case)
control.run()

## Tetrahedron and Hexahedron Elements on a 3D Beam

The beam starts at the origin, with length $L$ along $x$, width $W$ along $y$ and height $H$ along $z$.

### Trigonometric Function

$$\hat{\mathbf{u}}_R = \mathbf{R}\,(\mathbf{x} + \hat{\mathbf{u}}) - \mathbf{x}, \qquad \hat{\mathbf{P}}_R = \mathbf{R}\,\hat{\boldsymbol{\sigma}}, \qquad \mathbf{s}_R = 3k^2(\lambda + 2\mu)\,\mathbf{R}\,\hat{\mathbf{u}}, \qquad \mathbf{R} = \begin{pmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{pmatrix}$$

$$\hat{\mathbf{u}} = A\begin{pmatrix} \sin(kx)\cos(ky)\cos(kz) \\ \cos(kx)\sin(ky)\cos(kz) \\ \cos(kx)\cos(ky)\sin(kz) \end{pmatrix}$$

$$\hat{\boldsymbol{\sigma}} = Ak\begin{pmatrix} (3\lambda + 2\mu)\,\cos(kx)\cos(ky)\cos(kz) & -2\mu\,\sin(kx)\sin(ky)\cos(kz) & -2\mu\,\sin(kx)\cos(ky)\sin(kz) \\ -2\mu\,\sin(kx)\sin(ky)\cos(kz) & (3\lambda + 2\mu)\,\cos(kx)\cos(ky)\cos(kz) & -2\mu\,\cos(kx)\sin(ky)\sin(kz) \\ -2\mu\,\sin(kx)\cos(ky)\sin(kz) & -2\mu\,\cos(kx)\sin(ky)\sin(kz) & (3\lambda + 2\mu)\,\cos(kx)\cos(ky)\cos(kz) \end{pmatrix}$$

with $k = \frac{2\pi}{L}$, $L = W = H$, and $\theta = 45^\circ$.

- `bottom` ($y = 0$), `front` ($z = 0$), `back` ($z = L$): $\mathbf{u} = \hat{\mathbf{u}}_R$
- `left` ($x = 0$), `right` ($x = L$), `top` ($y = L$): $\mathbf{t} = \hat{\mathbf{P}}_R\,\mathbf{n}$

In [ ]:
# Case: 3D box rotated by 45° about z, trigonometric manufactured solution, tetrahedra (left) and hexahedra (right)
tet = run("../cases/CorotationalFEMForceField/Trigonometric3D_Tetra_ConstantRotation.json")
hexa = run("../cases/CorotationalFEMForceField/Trigonometric3D_Hexa_ConstantRotation.json")
figure, (tetrahedra, hexahedra) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tet.plot(tetrahedra)
hexa.plot(hexahedra)

# Deformed shape at each level, coarse to fine
tet.plot_fields()
hexa.plot_fields()

### Trigonometric Function with a Varying Rotation

$$\hat{\mathbf{u}}_R = \mathbf{R}(\theta)\,(\mathbf{x} + \hat{\mathbf{u}}) - \mathbf{x}, \qquad \theta = \alpha_x x + \alpha_y y, \qquad \hat{\boldsymbol{\varepsilon}}_U = \hat{\mathbf{R}}^T \hat{\mathbf{F}} - \mathbf{I}, \qquad \hat{\mathbf{P}}_R = \hat{\mathbf{R}}\,\boldsymbol{\sigma}(\hat{\boldsymbol{\varepsilon}}_U), \qquad \mathbf{s}_R = -\nabla \cdot \hat{\mathbf{P}}_R$$

$$\hat{\mathbf{F}} = \mathbf{I} + \nabla \hat{\mathbf{u}}_R = \mathbf{W}\,\boldsymbol{\Sigma}\,\mathbf{V}^T, \qquad \hat{\mathbf{R}} = \mathbf{W}\,\mathbf{V}^T$$

with $\hat{\mathbf{u}}$ and $\mathbf{R}$ as in the trigonometric case above, and $\alpha_x = 3.75^\circ$, $\alpha_y = 2.5^\circ$ per unit length, half the 2D rates: in 3D, under traction, the 2D rates bring the finest mesh close to buckling. $\hat{\mathbf{R}}$ and $\hat{\mathbf{P}}_R$ are computed at each point, and $\mathbf{s}_R$ by fourth-order central differences of $\hat{\mathbf{P}}_R$.

- `bottom` ($y = 0$), `front` ($z = 0$), `back` ($z = L$): $\mathbf{u} = \hat{\mathbf{u}}_R$
- `left` ($x = 0$), `right` ($x = L$), `top` ($y = L$): $\mathbf{t} = \hat{\mathbf{P}}_R\,\mathbf{n}$

In [ ]:
# Case: 3D box with a rotation field about z, trigonometric manufactured solution, tetrahedra (left) and hexahedra (right)
tet_field = run("../cases/CorotationalFEMForceField/Trigonometric3D_Tetra_VaryingRotation.json")
hexa_field = run("../cases/CorotationalFEMForceField/Trigonometric3D_Hexa_VaryingRotation.json")
figure, (tetrahedra, hexahedra) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tet_field.plot(tetrahedra)
hexa_field.plot(hexahedra)

# Deformed shape at each level, coarse to fine
tet_field.plot_fields()
hexa_field.plot_fields()

<a id="references"></a>
## References

<a id="ref1"></a>
**[1]** C. A. Felippa and B. Haugen, "A unified formulation of small-strain corotational finite elements: I. Theory," *Computer Methods in Applied Mechanics and Engineering*, 194(21–24):2285–2335, 2005.